In [8]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import clone

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

train_df = pd.read_csv("../data/training.csv")

print("Dataset Shape:", train_df.shape)
display(train_df.head())

X = train_df.drop(columns=["CustomerID", "Churn"])
y = train_df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)



Dataset Shape: (440833, 12)


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


Features shape: (440833, 10)
Target shape: (440833,)


In [3]:

numerical_features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]

categorical_features = [
    "Gender",
    "Subscription Type",
    "Contract Length"
]


In [4]:

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])


In [6]:

# Remove rows with missing target values
train_df = train_df.dropna(subset=["Churn"]).copy()

# Recreate features and target
X = train_df.drop(columns=["CustomerID", "Churn"])
y = train_df["Churn"].copy()

# Verify target
print("Missing values in y:", y.isna().sum())
print("Total rows:", len(y))

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Split successful!")

Missing values in y: 0
Total rows: 440832

Training data: (352665, 10)
Validation data: (88167, 10)
Split successful!


In [9]:
decision_tree_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

# Train
decision_tree_model.fit(X_train, y_train)

# Predict
dt_pred = decision_tree_model.predict(X_val)
dt_prob = decision_tree_model.predict_proba(X_val)[:, 1]

print("Decision Tree training completed!")

Decision Tree training completed!


In [10]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
random_forest_model.fit(X_train, y_train)

# Predict
rf_pred = random_forest_model.predict(X_val)
rf_prob = random_forest_model.predict_proba(X_val)[:, 1]

print("Random Forest training completed!")

Random Forest training completed!


In [11]:
gradient_boosting_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# Train
gradient_boosting_model.fit(X_train, y_train)

# Predict
gb_pred = gradient_boosting_model.predict(X_val)
gb_prob = gradient_boosting_model.predict_proba(X_val)[:, 1]

print("Gradient Boosting training completed!")

Gradient Boosting training completed!


In [12]:
def evaluate_model(model_name, y_true, y_pred, y_prob):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

In [14]:
results = []

# Decision Tree
results.append(
    evaluate_model(
        "Decision Tree",
        y_val,
        dt_pred,
        dt_prob
    )
)

# Random Forest
results.append(
    evaluate_model(
        "Random Forest",
        y_val,
        rf_pred,
        rf_prob
    )
)

# Gradient Boosting
results.append(
    evaluate_model(
        "Gradient Boosting",
        y_val,
        gb_pred,
        gb_prob
    )
)

# Display comparison
results_df = pd.DataFrame(results)

print(results_df.round(4))

               Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0      Decision Tree    0.9999     0.9999  0.9999    0.9999   0.9999
1      Random Forest    0.9992     0.9999  0.9986    0.9993   1.0000
2  Gradient Boosting    0.9967     0.9999  0.9943    0.9971   0.9999


In [15]:
models = {
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}

for name, model in models.items():

    train_accuracy = model.score(X_train, y_train)
    val_accuracy = model.score(X_val, y_val)

    print(f"\n{name}")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Difference: {train_accuracy - val_accuracy:.4f}")


Decision Tree
Training Accuracy: 1.0000
Validation Accuracy: 0.9999
Difference: 0.0001

Random Forest
Training Accuracy: 0.9999
Validation Accuracy: 0.9992
Difference: 0.0007

Gradient Boosting
Training Accuracy: 0.9967
Validation Accuracy: 0.9967
Difference: 0.0000


In [16]:
test_df = pd.read_csv("../data/testing.csv")

# Remove rows with missing target values, if any
test_df = test_df.dropna(subset=["Churn"]).copy()

# Separate features and target
X_test = test_df.drop(columns=["CustomerID", "Churn"])
y_test = test_df["Churn"]

print("Test data shape:", X_test.shape)
print("Test target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Test data shape: (64374, 10)
Test target distribution:
Churn
0    52.631497
1    47.368503
Name: proportion, dtype: float64


In [17]:
#save the trained models
import joblib

joblib.dump(decision_tree_model, "../models/decision_tree.pkl")
joblib.dump(random_forest_model, "../models/random_forest.pkl")
joblib.dump(gradient_boosting_model, "../models/gradient_boosting.pkl")

print("All models saved successfully!")

All models saved successfully!
